##overall CODE for training
model is hybrid (wave2vec for feature extarction and LSTM for calssification stress and emotion later) **bold text**

In [ ]:
!pip install transformers datasets librosa accelerate

In [ ]:
import torch
import torch.nn as nn
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor

In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")


In [ ]:
for param in wav2vec.feature_extractor.parameters():
    param.requires_grad = False


In [ ]:
class Wav2Vec2_LSTM_MultiTask(nn.Module):
    def __init__(self, num_emotions):
        super().__init__()

        self.wav2vec = wav2vec

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.shared_fc = nn.Linear(512, 256)

        # Emotion (classification)
        self.emotion_head = nn.Linear(256, num_emotions)

        # Stress (regression)
        self.stress_head = nn.Linear(256, 1)

    def forward(self, input_values):
        with torch.no_grad():
            outputs = self.wav2vec(input_values)

        x = outputs.last_hidden_state   # (B, T, 768)

        lstm_out, _ = self.lstm(x)
        pooled = torch.mean(lstm_out, dim=1)

        shared = torch.relu(self.shared_fc(pooled))

        emotion_logits = self.emotion_head(shared)
        stress_value = self.stress_head(shared)

        return emotion_logits, stress_value


In [ ]:
def load_audio(path):
    audio, sr = librosa.load(path, sr=16000)
    return audio


In [ ]:
def prepare_sample(audio_path):
    audio = load_audio(audio_path)
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    return inputs.input_values.squeeze(0)


In [ ]:
emotion_loss_fn = nn.CrossEntropyLoss()
stress_loss_fn = nn.MSELoss()


In [ ]:
def combined_loss(emotion_logits, stress_pred, emotion_gt, stress_gt):
    loss_emotion = emotion_loss_fn(emotion_logits, emotion_gt)
    loss_stress = stress_loss_fn(stress_pred.squeeze(), stress_gt)
    return loss_emotion + 0.5 * loss_stress


##load dataset from kaggle
# RAVDESS


In [ ]:
!pip install kaggle


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
!unzip ravdess-emotional-speech-audio.zip


In [ ]:
DATA_DIR = "/content"


In [ ]:
import os
import pandas as pd

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

stress_map = {
    "neutral": 0.2,
    "calm": 0.1,
    "happy": 0.3,
    "sad": 0.6,
    "fearful": 0.8,
    "angry": 0.9,
    "disgust": 0.7,
    "surprised": 0.5
}

rows = []

for root, _, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            emotion_label = emotion_map[emotion_code]

            rows.append({
                "audio_path": os.path.join(root, file),
                "emotion_label": emotion_label,
                "stress": stress_map[emotion_label]
            })

df = pd.DataFrame(rows)
print("Total samples:", len(df))
df.head()


In [ ]:
print(df.columns)


In [ ]:
emotion2id = {e: i for i, e in enumerate(df["emotion_label"].unique())}
df["emotion_id"] = df["emotion_label"].map(emotion2id)

NUM_EMOTIONS = len(emotion2id)
NUM_EMOTIONS


In [ ]:
print(df.columns)
df.sample(5)


In [ ]:
model = Wav2Vec2_LSTM_MultiTask(num_emotions=NUM_EMOTIONS)
model.cuda()


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["emotion_id"],
    random_state=42
)

print(len(train_df), len(test_df))


In [ ]:
import torch
from torch.utils.data import Dataset
import librosa
from transformers import Wav2Vec2Processor

# Make sure processor exists
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

def load_audio(path):
    audio, _ = librosa.load(path, sr=16000)
    return audio


class RAVDESSDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio = load_audio(row["audio_path"])
        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(0),
            "emotion": torch.tensor(row["emotion_id"], dtype=torch.long),
            "stress": torch.tensor(row["stress"], dtype=torch.float)
        }


In [ ]:
train_ds = RAVDESSDataset(train_df)
test_ds = RAVDESSDataset(test_df)


In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

def collate_fn(batch):
    input_values = pad_sequence(
        [b["input_values"] for b in batch],
        batch_first=True
    )

    emotions = torch.stack([b["emotion"] for b in batch])
    stress = torch.stack([b["stress"] for b in batch])

    return {
        "input_values": input_values,
        "emotion": emotions,
        "stress": stress
    }


In [ ]:
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=2, collate_fn=collate_fn)


In [ ]:
model = Wav2Vec2_LSTM_MultiTask(num_emotions=NUM_EMOTIONS)
model = model.cuda()


In [ ]:
import torch.nn as nn

emotion_loss_fn = nn.CrossEntropyLoss()
stress_loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)


In [ ]:
def total_loss_fn(em_logits, st_pred, em_gt, st_gt):
    loss_em = emotion_loss_fn(em_logits, em_gt)
    loss_st = stress_loss_fn(st_pred.squeeze(), st_gt)
    return loss_em + 0.5 * loss_st


In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_values = batch["input_values"].cuda()
        emotion_gt = batch["emotion"].cuda()
        stress_gt = batch["stress"].cuda()

        em_logits, st_pred = model(input_values)

        loss = total_loss_fn(
            em_logits, st_pred,
            emotion_gt, stress_gt
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_values = batch["input_values"].cuda()
        emotion_gt = batch["emotion"].cuda()

        em_logits, _ = model(input_values)
        preds = em_logits.argmax(dim=1)

        correct += (preds == emotion_gt).sum().item()
        total += len(emotion_gt)

print("Emotion Accuracy:", correct / total)


In [ ]:
import torch

preds_all = []
gt_all = []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch["input_values"].cuda()
        stress_gt = batch["stress"].cuda()

        _, stress_pred = model(input_values)

        preds_all.extend(stress_pred.squeeze().cpu().numpy())
        gt_all.extend(stress_gt.cpu().numpy())

rmse = ((torch.tensor(preds_all) - torch.tensor(gt_all))**2).mean().sqrt()
print("Stress RMSE:", rmse.item())


# 3 dataset download from kaggle

In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# RAVDESS
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
!unzip -q ravdess-emotional-speech-audio.zip

# TESS
!kaggle datasets download -d ejlok1/toronto-emotional-speech-set-tess
!unzip -q toronto-emotional-speech-set-tess.zip

# SAVEE
!kaggle datasets download -d ejlok1/surrey-audiovisual-expressed-emotion-savee
!unzip -q surrey-audiovisual-expressed-emotion-savee.zip


## all three dataset together training


In [ ]:
!pip install -q torch transformers librosa soundfile scikit-learn pandas


In [ ]:
import os
import torch
import librosa
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.model_selection import train_test_split
import torch.nn as nn


In [ ]:
COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear"]
emotion2id = {e: i for i, e in enumerate(COMMON_EMOTIONS)}
id2emotion = {i: e for e, i in emotion2id.items()}


In [ ]:
GLOBAL_MAP = {
    "neutral": "neutral",
    "calm": "neutral",
    "happy": "happy",
    "sad": "sad",
    "angry": "angry",
    "fear": "fear",
    "fearful": "fear",
    "disgust": None,
    "surprised": None
}


In [ ]:
ravdess_rows = []

ravdess_map = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"
}

for root, _, files in os.walk("/content/audio_speech_actors_01-24"):
    for f in files:
        if f.endswith(".wav"):
            raw = ravdess_map[f.split("-")[2]]
            emo = GLOBAL_MAP.get(raw)
            if emo:
                ravdess_rows.append({
                    "audio_path": os.path.join(root, f),
                    "emotion": emo
                })

df_ravdess = pd.DataFrame(ravdess_rows)
print("RAVDESS:", df_ravdess.shape)


In [ ]:
tess_rows = []
TESS_DIR = "/content/TESS Toronto emotional speech set data"

for root, _, files in os.walk(TESS_DIR):
    for f in files:
        if f.endswith(".wav"):
            raw = f.split("_")[-1].replace(".wav", "").lower()
            emo = GLOBAL_MAP.get(raw)
            if emo:
                tess_rows.append({
                    "audio_path": os.path.join(root, f),
                    "emotion": emo
                })

df_tess = pd.DataFrame(tess_rows)
print("TESS:", df_tess.shape)


In [ ]:
savee_rows = []
SAVE_DIR = "/content/ALL"

savee_map = {
    "n": "neutral", "h": "happy", "sa": "sad",
    "a": "angry", "f": "fear"
}

for f in os.listdir(SAVE_DIR):
    if f.endswith(".wav"):
        code = f.split("_")[1][:2]
        raw = savee_map.get(code)
        emo = GLOBAL_MAP.get(raw)
        if emo:
            savee_rows.append({
                "audio_path": os.path.join(SAVE_DIR, f),
                "emotion": emo
            })

df_savee = pd.DataFrame(savee_rows)
print("SAVEE:", df_savee.shape)


In [ ]:
#merge model
df_all = pd.concat([df_ravdess, df_tess, df_savee], ignore_index=True)

df_all["emotion_id"] = df_all["emotion"].map(emotion2id)

stress_map = {
    "neutral": 0.2,
    "happy": 0.3,
    "sad": 0.6,
    "fear": 0.8,
    "angry": 0.9
}
df_all["stress"] = df_all["emotion"].map(stress_map)

print(df_all["emotion"].value_counts())
print("TOTAL:", len(df_all))


In [ ]:
train_df, test_df = train_test_split(
    df_all,
    test_size=0.2,
    stratify=df_all["emotion_id"],
    random_state=42
)


In [ ]:
print(train_df["emotion"].value_counts())
print(test_df["emotion"].value_counts())
print(train_df["stress"].value_counts())
print(test_df["stress"].value_counts())


In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

class UnifiedSpeechDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, _ = librosa.load(row["audio_path"], sr=16000)

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        )

        return {
            "input_values": inputs.input_values.squeeze(0),
            "emotion": torch.tensor(row["emotion_id"]),
            "stress": torch.tensor(row["stress"], dtype=torch.float)
        }


In [ ]:
def collate_fn(batch):
    input_values = pad_sequence(
        [b["input_values"] for b in batch],
        batch_first=True
    )
    emotions = torch.stack([b["emotion"] for b in batch])
    stress = torch.stack([b["stress"] for b in batch])

    return {
        "input_values": input_values,
        "emotion": emotions,
        "stress": stress
    }


In [ ]:
#data loader
train_loader = DataLoader(
    UnifiedSpeechDataset(train_df),
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    UnifiedSpeechDataset(test_df),
    batch_size=2,
    collate_fn=collate_fn
)


In [ ]:
#hybrid model
class Wav2Vec2_LSTM_MultiTask(nn.Module):
    def __init__(self, num_emotions):
        super().__init__()

        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(512, 256)
        self.emotion_head = nn.Linear(256, num_emotions)
        self.stress_head = nn.Linear(256, 1)

    def forward(self, input_values):
        x = self.wav2vec(input_values).last_hidden_state
        x, _ = self.lstm(x)
        x = torch.mean(x, dim=1)
        x = torch.relu(self.fc(x))
        return self.emotion_head(x), self.stress_head(x)


In [ ]:
#training code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Wav2Vec2_LSTM_MultiTask(len(COMMON_EMOTIONS)).to(device)

for p in model.wav2vec.parameters():
    p.requires_grad = False

emotion_loss = nn.CrossEntropyLoss()
stress_loss = nn.MSELoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4
)


In [ ]:
#train loop
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total = 0

    for batch in train_loader:
        inputs = batch["input_values"].to(device)
        emo_gt = batch["emotion"].to(device)
        stress_gt = batch["stress"].to(device)

        emo_logits, stress_pred = model(inputs)

        loss = emotion_loss(emo_logits, emo_gt) + \
               0.5 * stress_loss(stress_pred.squeeze(), stress_gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total/len(train_loader):.4f}")


## Save Model

In [ ]:
from google.colab import drive
drive.mount('/content/mydrive')



In [ ]:
SAVE_DIR = "/content/mydrive/MyDrive/hybrid_wav2vec2_lstm"
import os
os.makedirs(SAVE_DIR, exist_ok=True)



In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "emotion2id": emotion2id,
    "num_emotions": NUM_EMOTIONS
}, os.path.join(SAVE_DIR, "model.pt"))

print("Model saved at:", SAVE_DIR)


In [ ]:
processor.save_pretrained(SAVE_DIR)


In [ ]:
# Create main directory
!mkdir -p /content/drive/MyDrive/wav2vec2_hybrid_model

# Create required files
!touch /content/drive/MyDrive/wav2vec2_hybrid_model/model.pt
!touch /content/drive/MyDrive/wav2vec2_hybrid_model/model.py
!touch /content/drive/MyDrive/wav2vec2_hybrid_model/app.py
!touch /content/drive/MyDrive/wav2vec2_hybrid_model/requirements.txt
!touch /content/drive/MyDrive/wav2vec2_hybrid_model/README.md

# Verify structure
!tree /content/drive/MyDrive/wav2vec2_hybrid_model

#push model to hugging face

In [ ]:
from huggingface_hub import login, upload_folder

login()

upload_folder(
    folder_path="/content/mydrive/MyDrive/hybrid_wav2vec2_lstm",
    repo_id="ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS",
    repo_type="model"
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...id_wav2vec2_lstm/model.pt:   9%|8         | 33.6MB /  393MB            

CommitInfo(commit_url='https://huggingface.co/ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS/commit/3b9c1dfb90b6ecf5b99ba9aac4b06bb1ce60d780', commit_message='Upload folder using huggingface_hub', commit_description='', oid='3b9c1dfb90b6ecf5b99ba9aac4b06bb1ce60d780', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS', endpoint='https://huggingface.co', repo_type='model', repo_id='ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS'), pr_revision=None, pr_num=None)

In [ ]:
!mkdir -p /content/mydrive/MyDrive/hybrid_wav2vec2_lstm


In [ ]:
!touch /content/mydrive/MyDrive/hybrid_wav2vec2_lstm/model.py
!touch /content/mydrive/MyDrive/hybrid_wav2vec2_lstm/inference.py
!touch /content/mydrive/MyDrive/hybrid_wav2vec2_lstm/requirements.txt
!touch /content/mydrive/MyDrive/hybrid_wav2vec2_lstm/README.md


In [ ]:
import os

BASE_DIR = "/content/mydrive/MyDrive/hybrid_wav2vec2_lstm"
os.makedirs(BASE_DIR, exist_ok=True)

print("Folder ready:", BASE_DIR)


Folder ready: /content/mydrive/MyDrive/hybrid_wav2vec2_lstm


In [ ]:
model_code = """
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model

class Wav2Vec2_LSTM_MultiTask(nn.Module):
    def __init__(self, num_emotions):
        super().__init__()

        self.wav2vec = Wav2Vec2Model.from_pretrained(
            "facebook/wav2vec2-base"
        )

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.shared_fc = nn.Linear(512, 256)
        self.emotion_head = nn.Linear(256, num_emotions)
        self.stress_head = nn.Linear(256, 1)

    def forward(self, input_values):
        outputs = self.wav2vec(input_values)
        x = outputs.last_hidden_state

        lstm_out, _ = self.lstm(x)
        pooled = torch.mean(lstm_out, dim=1)

        shared = torch.relu(self.shared_fc(pooled))

        emotion_logits = self.emotion_head(shared)
        stress_value = self.stress_head(shared)

        return emotion_logits, stress_value
"""

with open(os.path.join(BASE_DIR, "model.py"), "w") as f:
    f.write(model_code)

print("model.py written")


model.py written


In [ ]:
inference_code = """
import torch
import librosa
from transformers import Wav2Vec2Processor
from model import Wav2Vec2_LSTM_MultiTask

checkpoint = torch.load("model.pt", map_location="cpu")

emotion2id = checkpoint["emotion2id"]
id2emotion = {v: k for k, v in emotion2id.items()}
NUM_EMOTIONS = checkpoint["num_emotions"]

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2_LSTM_MultiTask(NUM_EMOTIONS)
model.load_state_dict(checkpoint["model_state"])
model.eval()

def predict(audio_path):
    audio, _ = librosa.load(audio_path, sr=16000)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_values

    with torch.no_grad():
        emotion_logits, stress_pred = model(inputs)

    return {
        "emotion": id2emotion[emotion_logits.argmax(dim=1).item()],
        "stress": round(stress_pred.item(), 3)
    }
"""

with open(os.path.join(BASE_DIR, "inference.py"), "w") as f:
    f.write(inference_code)

print("inference.py written")


inference.py written


In [ ]:
requirements = """torch
transformers
librosa
soundfile
numpy
"""

with open(os.path.join(BASE_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

print("requirements.txt written")


requirements.txt written


In [ ]:
readme = """# Hybrid Wav2Vec2 + LSTM Emotion & Stress Model

This repository contains a trained hybrid Wav2Vec2 + LSTM model
for emotion and stress recognition from raw speech audio.

- Input: WAV audio (16 kHz)
- Output: Emotion label + Stress score

Training datasets:
- RAVDESS
- TESS
- SAVEE

Inference code is included.
"""

with open(os.path.join(BASE_DIR, "README.md"), "w") as f:
    f.write(readme)

print("README.md written")


README.md written


In [ ]:
os.listdir(BASE_DIR)


['model.pt',
 'preprocessor_config.json',
 'tokenizer_config.json',
 'special_tokens_map.json',
 'vocab.json',
 'model.py',
 'inference.py',
 'requirements.txt',
 'README.md']

In [ ]:
from huggingface_hub import login, upload_folder

login()

upload_folder(
    folder_path=BASE_DIR,
    repo_id="ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS",
    repo_type="model"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...id_wav2vec2_lstm/model.pt:   4%|4         | 16.7MB /  393MB            

CommitInfo(commit_url='https://huggingface.co/ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS/commit/ec5ddac111ebaef34c8f35e9ac4e955fd0db5529', commit_message='Upload folder using huggingface_hub', commit_description='', oid='ec5ddac111ebaef34c8f35e9ac4e955fd0db5529', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS', endpoint='https://huggingface.co', repo_type='model', repo_id='ashutoshroy02/hybrid-wave2vec-LSTM-emotion-stress-RAVDESS'), pr_revision=None, pr_num=None)

## testing on real audio

In [ ]:
import torch
import librosa
from transformers import Wav2Vec2Processor

# -----------------------------
# 1. Paths
# -----------------------------
MODEL_DIR = "/content/mydrive/MyDrive/hybrid_wav2vec2_lstm"
MODEL_PATH = f"{MODEL_DIR}/model.pt"

# -----------------------------
# 2. Load processor
# -----------------------------
processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)

# -----------------------------
# 3. Recreate model architecture
# -----------------------------
checkpoint = torch.load(MODEL_PATH)

emotion2id = checkpoint["emotion2id"]
NUM_EMOTIONS = checkpoint["num_emotions"]
id2emotion = {v: k for k, v in emotion2id.items()}

model = Wav2Vec2_LSTM_MultiTask(num_emotions=NUM_EMOTIONS)
model.load_state_dict(checkpoint["model_state"])
model = model.cuda()
model.eval()

print("✅ Model loaded successfully")

# -----------------------------
# 4. Audio loader
# -----------------------------
def load_audio(path):
    audio, _ = librosa.load(path, sr=16000)
    return audio

# -----------------------------
# 5. Predict function
# -----------------------------
def predict_audio(audio_path):
    audio = load_audio(audio_path)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_values.cuda()

    with torch.no_grad():
        emotion_logits, stress_pred = model(inputs)

    emotion_id = emotion_logits.argmax(dim=1).item()
    emotion = id2emotion[emotion_id]
    stress = stress_pred.item()

    return emotion, round(stress, 3)

# -----------------------------
# 6. TEST WITH YOUR AUDIO
# -----------------------------
emotion, stress = predict_audio("/content/drive/MyDrive/test.wav")

print("🎭 Predicted Emotion:", emotion)
print("😰 Predicted Stress Level:", stress)


✅ Model loaded successfully
🎭 Predicted Emotion: sad
😰 Predicted Stress Level: 0.496


# test on streamlit with live audio colab

In [ ]:
import os

APP_PATH = "/content/app.py"

streamlit_code = """
import streamlit as st
import torch
import librosa
from transformers import Wav2Vec2Processor
from model import Wav2Vec2_LSTM_MultiTask
import tempfile

st.set_page_config(page_title="Emotion & Stress Detection")

st.title("🎤 Emotion & Stress Detection (Upload Audio)")

# ------------------------
# Load model
# ------------------------
MODEL_DIR = "/content/mydrive/MyDrive/hybrid_wav2vec2_lstm"

checkpoint = torch.load(f"{MODEL_DIR}/model.pt", map_location="cpu")
emotion2id = checkpoint["emotion2id"]
id2emotion = {v: k for k, v in emotion2id.items()}
NUM_EMOTIONS = checkpoint["num_emotions"]

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2_LSTM_MultiTask(NUM_EMOTIONS)
model.load_state_dict(checkpoint["model_state"])
model.eval()

# ------------------------
# Upload audio
# ------------------------
uploaded_file = st.file_uploader("Upload a WAV file", type=["wav"])

if uploaded_file is not None:
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        tmp.write(uploaded_file.read())
        audio_path = tmp.name

    audio, _ = librosa.load(audio_path, sr=16000)

    st.audio(audio_path)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_values

    with torch.no_grad():
        emotion_logits, stress_pred = model(inputs)

    emotion = id2emotion[emotion_logits.argmax(dim=1).item()]
    stress = round(stress_pred.item(), 3)

    st.success(f"🎭 Emotion: {emotion}")
    st.warning(f"😰 Stress Level: {stress}")

"""

with open(APP_PATH, "w") as f:
    f.write(streamlit_code)

print("✅ Streamlit app written to", APP_PATH)


✅ Streamlit app written to /content/app.py


In [ ]:
!streamlit run app.py --server.port 8501 --server.enableCORS false



2025-12-14 22:35:45.932 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.169.12.126:8501

  Stopping...
  Stopping...


In [ ]:
!pip install gradio


In [ ]:
import gradio as gr
import torch
import librosa
from transformers import Wav2Vec2Processor
from model import Wav2Vec2_LSTM_MultiTask

MODEL_DIR = "/content/mydrive/MyDrive/hybrid_wav2vec2_lstm"
device = torch.device("cpu")

checkpoint = torch.load(
    f"{MODEL_DIR}/model.pt",
    map_location=device
)

emotion2id = checkpoint["emotion2id"]
id2emotion = {v: k for k, v in emotion2id.items()}
NUM_EMOTIONS = checkpoint["num_emotions"]

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2_LSTM_MultiTask(NUM_EMOTIONS)
model.load_state_dict(checkpoint["model_state"])
model.to(device)
model.eval()

def predict(audio):
    try:
        if audio is None:
            return "No audio", 0.0

        sr, audio = audio
        if audio is None or len(audio) < 1600:
            return "Audio too short", 0.0

        audio = audio.astype("float32")
        audio = librosa.resample(audio, sr, 16000)

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        ).input_values.to(device)

        with torch.no_grad():
            emotion_logits, stress_pred = model(inputs)

        emotion = id2emotion[int(torch.argmax(emotion_logits, dim=1))]
        stress = float(stress_pred.squeeze().item())

        return emotion, round(stress, 3)

    except Exception as e:
        return f"Error: {e}", 0.0

gr.Interface(
    fn=predict,
    inputs=gr.Audio(type="numpy"),
    outputs=[
        gr.Textbox(label="Emotion"),
        gr.Number(label="Stress Level")
    ],
    title="🎤 Live Emotion & Stress Detection"
).launch(share=True)


/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://821e262734016b83a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
